<a href="https://colab.research.google.com/github/chian0501/OmiNote/blob/main/Life.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 0: 資料庫初始化 (若 lifedb.db 不存在則自動建立)

# 2. 環境設定與套件安裝 (Cell 1)
這段代碼負責安裝必要的 Python 套件 (bidict, lunar_python 等)，並從 config.py 載入設定參數。

In [1]:
# Cell 1: 基礎設定 (安裝、掛載、載入設定檔與資料庫)
# ============================================================================
print("--- 步驟 1/4: 正在檢查並安裝必要套件... ---")
!pip install -q bidict sxtwl lunar_python colorama py-iztro kerykeion opencc-python-reimplemented ipywidgets
print("✅ 套件準備就緒")

print("\n--- 步驟 2/4: 設定環境路徑... ---")
from google.colab import drive
from pathlib import Path
import sys, importlib, sqlite3

if not Path('/content/drive').exists():
    drive.mount('/content/drive')

MODULES_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Life_DB")
if not MODULES_DIR.exists():
    MODULES_DIR.mkdir(parents=True, exist_ok=True)
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))

print("\n--- 步驟 3/4: 載入設定... ---")
try:
    import config
    importlib.reload(config)
    from config import TAIWAN_CITY_COORDS, PATHS, CUSTOM_FIXES, CITIES
    print("✅ 成功載入 config.py")
except ImportError:
    print("⚠️ 使用預設路徑設定")
    PATHS = {'base_dir': MODULES_DIR, 'output_dir': MODULES_DIR / "output", 'bazi_script': MODULES_DIR / "bazi.py"}
    CUSTOM_FIXES = {}

print("\n--- 步驟 4/4: 載入使用者資料庫... ---")
DB_PATH = PATHS['base_dir'] / "lifedb.db"
def get_all_users_from_db() -> dict:
    if not DB_PATH.exists(): return {}
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='users';")
        if not cursor.fetchone(): return {}
        cursor.execute("SELECT * FROM users ORDER BY username ASC;")
        return {row['username']: dict(row) for row in cursor.fetchall()}

USER_DB = get_all_users_from_db()
print(f"✅ 資料庫載入完成，共 {len(USER_DB)} 位使用者")

--- 步驟 1/4: 正在檢查並安裝必要套件... ---
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 1.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 80.6 MB/s eta 0:

# 3. 核心分析函式庫 (Cell 2)
這是最龐大的一塊，包含了：
八字分析 (使用 bazi.py)
紫微斗數分析 (使用 py-iztro)
西洋占星分析 (使用 kerykeion)
合併報告功能 (merge_user_reports)：負責將上述結果整合。

In [2]:
# Cell 2: 核心分析函式庫 (八字、紫微、合併報告) - 修復合併問題版
# ============================================================================
import subprocess, json, re, shutil, math
from datetime import datetime, date
from typing import List, Dict
from pathlib import Path

try:
    from py_iztro import Astro
    HAS_PY_IZTRO = True
except ImportError:
    HAS_PY_IZTRO = False
try:
    from opencc import OpenCC
    cc = OpenCC("s2twp")
    OPENCC_AVAILABLE = True
except ImportError:
    OPENCC_AVAILABLE = False

# --- 工具函式 ---
def safe_filename(name): return re.sub(r'[<>:"/\\|?*\x00-\x1f]', '_', name).strip()
def strip_ansi_codes(text): return re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])').sub('', text) if text else ""
def hour_to_time_index(hour): return 0 if hour == 23 else (hour + 1) // 2
def convert_and_fix(text):
    if not text: return ""
    if OPENCC_AVAILABLE: text = cc.convert(text)
    if 'CUSTOM_FIXES' in globals():
        for wrong, correct in CUSTOM_FIXES.items(): text = text.replace(wrong, correct)
    return text

# --- 八字分析 ---
def run_bazi_analysis(username: str, export: bool = True) -> str:
    if username not in USER_DB: return ""
    user = USER_DB[username]
    print(f"📊 執行八字: {user['name']}")
    args = [str(user['year']), str(user['month']), str(user['day']), str(user['hour']), '-g']
    if user['gender'] == "女": args.append("-n")
    bazi_script = PATHS.get("bazi_script")
    if not bazi_script or not bazi_script.exists(): return ""
    try:
        result = subprocess.run(["python", str(bazi_script)] + args, capture_output=True, text=True, check=True, encoding='utf-8', cwd=bazi_script.parent)
        bazi_result = convert_and_fix(strip_ansi_codes(result.stdout))
        if export:
            safe_name = safe_filename(user['name'])
            p = PATHS["output_dir"] / safe_name
            p.mkdir(parents=True, exist_ok=True)
            (p / f"{safe_name}_八字分析.txt").write_text(bazi_result, encoding="utf-8")
            print("✅ 八字已儲存")
        return bazi_result
    except: return ""

# --- 紫微斗數 ---
SI_HUA_MAP = {"甲": ["廉貞", "破軍", "武曲", "太陽"], "乙": ["天機", "天梁", "紫微", "太陰"], "丙": ["天同", "天機", "文昌", "廉貞"], "丁": ["太陰", "天同", "天機", "巨門"], "戊": ["貪狼", "太陰", "右弼", "天機"], "己": ["武曲", "貪狼", "天梁", "文曲"], "庚": ["太陽", "武曲", "太陰", "天同"], "辛": ["巨門", "太陽", "文曲", "文昌"], "壬": ["天梁", "紫微", "左輔", "武曲"], "癸": ["破軍", "巨門", "太陰", "貪狼"]}
def calculate_flying_stars(data):
    palaces = data.get("palaces", [])
    s_map = {s["name"]: p["name"] for p in palaces for s in p.get("majorStars", []) + p.get("minorStars", [])}
    flying = {}
    mutagens = ["祿", "權", "科", "忌"]
    for p in palaces:
        stem = p["heavenlyStem"]
        stars = SI_HUA_MAP.get(stem, [])
        targets = [{"star": s, "mutagen": mutagens[i], "to_palace": s_map.get(s, "未知")} for i, s in enumerate(stars)]
        flying[p["name"]] = {"stem": stem, "targets": targets}
    return flying

def run_ziwei_analysis(username, export=True):
    if not HAS_PY_IZTRO or username not in USER_DB: return
    user = USER_DB[username]
    try:
        dt = datetime(user['year'], user['month'], user['day'], user['hour'], user['minute'])
        gender = "男" if user['gender'] in ["M", "male", "男"] else "女"
        natal = Astro().by_solar(f"{dt.year}-{dt.month}-{dt.day}", hour_to_time_index(dt.hour), gender, language="zh-TW")
        data = json.loads(natal.model_dump_json(by_alias=True))
        data["extra_analysis"] = {"flying_stars": calculate_flying_stars(data)}
        if export:
            safe_name = safe_filename(user['name'])
            p = PATHS["output_dir"] / safe_name
            p.mkdir(parents=True, exist_ok=True)
            (p / f"{safe_name}_紫微本命.txt").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
            print("✅ 紫微本命完成")
    except Exception as e: print(f"Ziwei Error: {e}")

def run_ziwei_annual_transit(username, year, export=True):
    if not HAS_PY_IZTRO or username not in USER_DB: return
    user = USER_DB[username]
    try:
        dt = datetime(user['year'], user['month'], user['day'], user['hour'], user['minute'])
        gender = "男" if user['gender'] in ["M", "male", "男"] else "女"
        astro = Astro().by_solar(f"{dt.year}-{dt.month}-{dt.day}", hour_to_time_index(dt.hour), gender, language="zh-TW")
        transit = json.loads(astro.horoscope(f"{year}-01-01").model_dump_json(by_alias=True))
        if export:
            safe_name = safe_filename(user['name'])
            p = PATHS["output_dir"] / safe_name
            p.mkdir(parents=True, exist_ok=True)
            (p / f"{safe_name}_紫微流年_{year}.txt").write_text(json.dumps(transit, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"✅ 紫微流年 {year} 完成")
    except: pass

def run_ziwei_monthly_transit(username, year, month, export=True):
    if not HAS_PY_IZTRO or username not in USER_DB: return
    user = USER_DB[username]
    try:
        dt = datetime(user['year'], user['month'], user['day'], user['hour'], user['minute'])
        gender = "男" if user['gender'] in ["M", "male", "男"] else "女"
        astro = Astro().by_solar(f"{dt.year}-{dt.month}-{dt.day}", hour_to_time_index(dt.hour), gender, language="zh-TW")
        transit = json.loads(astro.horoscope(f"{year}-{month:02d}-01").model_dump_json(by_alias=True))
        if export:
            safe_name = safe_filename(user['name'])
            p = PATHS["output_dir"] / safe_name / "monthly_transits"
            p.mkdir(parents=True, exist_ok=True)
            (p / f"{safe_name}_紫微流月_{year}_{month:02d}.txt").write_text(json.dumps(transit, ensure_ascii=False, indent=2), encoding="utf-8")
    except: pass

# --- ★★★ 報告合併功能 (修復版) ★★★ ---
def merge_user_reports(username: str, custom_text: str = ""):
    if username not in USER_DB: return
    user = USER_DB[username]
    safe_name = safe_filename(user['name'])
    user_dir = PATHS["output_dir"] / safe_name
    if not user_dir.exists(): return

    print(f"📑 正在合併報告: {user['name']}...")
    raw_dir = user_dir / "原始檔案_Raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    all_files = list(user_dir.glob("*.txt")) + list((user_dir/"monthly_transits").glob("*.txt") if (user_dir/"monthly_transits").exists() else [])

    natal_files, transit_map, files_to_move = [], {}, []
    kw_natal = ["八字", "紫微本命", "西洋本命"]
    kw_transit = ["流年", "流月", "行運", "Transit", "行運分析"] # 增加關鍵字

    for f in all_files:
        if "綜合報告" in f.name: continue
        files_to_move.append(f)

        # 1. 判斷本命
        if any(k in f.name for k in kw_natal):
            natal_files.append(f)

        # 2. 判斷流年 (修復：更寬鬆的年份抓取)
        elif any(k in f.name for k in kw_transit):
            # 抓取檔名中任何位置的 4 位數 (例如 2025)
            match = re.search(r'(\d{4})', f.name)
            if match:
                y = match.group(1)
                transit_map.setdefault(y, []).append(f)
                # print(f"  -> 歸類為 {y} 年流年: {f.name}") # 除錯用
            else:
                print(f"  ⚠️ 無法識別年份: {f.name}")

    # 合併本命
    if natal_files:
        c = [f"# {user['name']} 本命綜合報告", f"日期: {datetime.now().strftime('%Y-%m-%d')}", "="*50, ""]
        for f in sorted(natal_files, key=lambda x: x.name):
            c.append(f"\n【{f.name}】\n{'-'*30}\n{f.read_text(encoding='utf-8')}\n{'='*50}")
        if custom_text: c.append(f"\n【備註】\n{custom_text}")
        (user_dir / f"{safe_name}_本命綜合報告.txt").write_text("\n".join(c), encoding='utf-8')

    # 合併流年
    for y, files in transit_map.items():
        c = [f"# {user['name']} {y}年 流年運勢", f"日期: {datetime.now().strftime('%Y-%m-%d')}", "="*50, ""]
        # 排序權重：紫微(0) -> 西洋行運(1) -> 行運分析(1) -> 流月(2) -> 其他
        def sort_key(x):
            if "紫微流年" in x.name: return 0
            if "西洋行運" in x.name or "行運分析" in x.name: return 1
            if "流月" in x.name: return 2
            return 3

        files.sort(key=lambda x: (sort_key(x), x.name))

        for f in files:
            c.append(f"\n【{f.name}】\n{'-'*30}\n{f.read_text(encoding='utf-8')}\n{'='*50}")
        if custom_text: c.append(f"\n【叮嚀】\n{custom_text}")
        (user_dir / f"{safe_name}_{y}年_流年綜合報告.txt").write_text("\n".join(c), encoding='utf-8')
        print(f"  ✅ 已建立 {y} 年流年報告")

    # 歸檔
    for f in files_to_move:
        try: shutil.move(str(f), str(raw_dir / f.name))
        except: pass
    if (user_dir/"monthly_transits").exists(): shutil.rmtree(user_dir/"monthly_transits")

In [3]:
# Cell 3: 西洋占星系統 (Kerykeion v5.0 優化整合版)
# ============================================================================
from kerykeion import AstrologicalSubjectFactory, ChartDataFactory
from datetime import datetime, timedelta
import re

# 1. 字典與設定
SIGN_ZH = {'Ari': '白羊', 'Tau': '金牛', 'Gem': '雙子', 'Can': '巨蟹', 'Leo': '獅子', 'Vir': '處女', 'Lib': '天秤', 'Sco': '天蠍', 'Sag': '射手', 'Cap': '摩羯', 'Aqu': '水瓶', 'Pis': '雙魚'}
PLANET_ZH = {'Sun': '太陽', 'Moon': '月亮', 'Mercury': '水星', 'Venus': '金星', 'Mars': '火星', 'Jupiter': '木星', 'Saturn': '土星', 'Uranus': '天王星', 'Neptune': '海王星', 'Pluto': '冥王星', 'Chiron': '凱龍星', 'True_Node': '北交點', 'South_Node': '南交點', 'Mean_Lilith': '莉莉絲', 'Ascendant': '上升', 'Medium_Coeli': '天頂'}
ASPECT_SYMBOL = {'conjunction': '☌', 'opposition': '☍', 'trine': '△', 'square': '□', 'sextile': '⚹'}
HOUSE_MAP = {'First_House': 1, 'Second_House': 2, 'Third_House': 3, 'Fourth_House': 4, 'Fifth_House': 5, 'Sixth_House': 6, 'Seventh_House': 7, 'Eighth_House': 8, 'Ninth_House': 9, 'Tenth_House': 10, 'Eleventh_House': 11, 'Twelfth_House': 12}
SIGN_INFO = {'Ari':('火','啟動'), 'Leo':('火','固定'), 'Sag':('火','變動'), 'Tau':('土','固定'), 'Vir':('土','變動'), 'Cap':('土','啟動'), 'Gem':('風','變動'), 'Lib':('風','啟動'), 'Aqu':('風','固定'), 'Can':('水','啟動'), 'Sco':('水','固定'), 'Pis':('水','變動')}
ORB_SETTINGS = {'outer': 0.5, 'inner': 0.3}

def fmt_deg(deg): return f"{int(deg):>2}°{int((deg-int(deg))*60):02d}'"
def get_sign_from_degree(degree):
    signs = ['Ari', 'Tau', 'Gem', 'Can', 'Leo', 'Vir', 'Lib', 'Sco', 'Sag', 'Cap', 'Aqu', 'Pis']
    return signs[int(degree/30)%12], degree%30

# ============================================================================
# 功能 A: 西洋本命分析 (純資料版)
# ============================================================================
def run_western_astrology(username: str, export: bool = True):
    if username not in USER_DB: return
    user = USER_DB[username]
    print(f"⭐ 執行西洋本命 (純資料): {user['name']}")

    subject = AstrologicalSubjectFactory.from_birth_data(
        name=user['name'], year=user['year'], month=user['month'], day=user['day'],
        hour=user['hour'], minute=user['minute'], lng=user['lng'], lat=user['lat'],
        tz_str=user['tz_str'], online=False
    )
    chart = ChartDataFactory.create_natal_chart_data(subject)

    lines = [f"【西洋占星本命盤】", f"姓名: {user['name']}", f"時間: {user['year']}-{user['month']:02d}-{user['day']:02d}", "="*60]

    # 1. 行星
    lines.append("【行星位置】")
    p_list = ['Sun', 'Moon', 'Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune', 'Pluto', 'Chiron', 'True_Node', 'Mean_Lilith', 'Ascendant', 'Medium_Coeli']
    elem_count = {'火':0, '土':0, '風':0, '水':0}

    for p in p_list:
        obj = getattr(subject, p.lower(), None)
        if obj:
            h_str = str(HOUSE_MAP.get(obj.house, "")) if hasattr(obj, 'house') else ""
            retro = "R" if getattr(obj, 'retrograde', False) else ""
            lines.append(f"  {PLANET_ZH.get(p,p):<8} {SIGN_ZH.get(obj.sign,obj.sign):<6} {fmt_deg(obj.position):<8} {h_str:<4} {retro}")
            if p in ['Sun','Moon','Mercury','Venus','Mars','Jupiter','Saturn'] and obj.sign in SIGN_INFO:
                elem_count[SIGN_INFO[obj.sign][0]] += 1

    # 南交點
    tn = getattr(subject, 'true_node', None)
    if tn:
        sn_sign, sn_deg = get_sign_from_degree((tn.position + 180) % 360)
        lines.append(f"  {'南交點':<8} {SIGN_ZH.get(sn_sign,sn_sign):<6} {fmt_deg(sn_deg):<8}")

    # 2. 宮位
    lines.append("-" * 30 + "\n【宮位系統】")
    houses = getattr(subject, 'houses', [])
    for i, h_deg in enumerate(houses):
        h_sign, h_rel = get_sign_from_degree(h_deg)
        lines.append(f"  第{i+1:<2}宮    {SIGN_ZH.get(h_sign,h_sign):<6} {fmt_deg(h_rel)}")

    # 3. 相位 (依容許度排序)
    lines.append("=" * 60 + "\n【相位列表】")
    valid_aspects = [asp for asp in chart.aspects if asp.aspect in ['conjunction', 'opposition', 'square', 'trine', 'sextile']]
    valid_aspects.sort(key=lambda x: abs(x.orbit))
    for asp in valid_aspects:
        lines.append(f"  {PLANET_ZH.get(asp.p1_name,asp.p1_name):<8} {ASPECT_SYMBOL.get(asp.aspect,''):<4} {PLANET_ZH.get(asp.p2_name,asp.p2_name):<8} {abs(asp.orbit):.2f}°")

    content = "\n".join(lines)
    if export:
        safe_name = safe_filename(user['name'])
        p = PATHS["output_dir"] / safe_name
        p.mkdir(parents=True, exist_ok=True)
        (p / f"{safe_name}_西洋本命_純資料.txt").write_text(content, encoding="utf-8")
        print("✅ 西洋本命完成")

# ============================================================================
# 功能 B: 西洋行運分析 (v5.0 優化版 - 強化篩選)
# ============================================================================
def create_transit_chart(date, user_data):
    return AstrologicalSubjectFactory.from_birth_data(
        name=f"Transit_{date.strftime('%Y%m%d')}", year=date.year, month=date.month, day=date.day,
        hour=date.hour, minute=date.minute, lng=user_data['lng'], lat=user_data['lat'],
        tz_str=user_data.get('tz_str', 'Asia/Taipei'), online=False
    )

def calculate_transit_aspects(natal, transit):
    chart = ChartDataFactory.create_transit_chart_data(natal, transit)
    filtered = []
    for asp in chart.aspects:
        t_name, n_name = asp.p1_name.lower(), asp.p2_name.lower()
        if t_name == 'moon': continue # 忽略行運月亮
        if any(x in t_name or x in n_name for x in ['node','lilith','chiron']): continue # 忽略虛點

        # 篩選規則：只看木土天海冥，或是日水金的合/沖
        is_outer = t_name in ['jupiter','saturn','uranus','neptune','pluto']
        is_inner_major = t_name in ['sun','mercury','venus'] and asp.aspect in ['conjunction','opposition']

        if is_outer or is_inner_major:
            limit = ORB_SETTINGS['outer'] if is_outer else ORB_SETTINGS['inner']
            if abs(asp.orbit) <= limit:
                filtered.append(asp)
    return filtered

def run_western_transit(username: str, year: int = 2025, export: bool = True):
    if username not in USER_DB: return
    user = USER_DB[username]
    print(f"⭐ 執行西洋行運分析 ({year}): {user['name']}")

    natal = AstrologicalSubjectFactory.from_birth_data(
        name=user['name'], year=user['year'], month=user['month'], day=user['day'],
        hour=user['hour'], minute=user['minute'], lng=user['lng'], lat=user['lat'],
        tz_str=user['tz_str'], online=False
    )

    start_date = datetime(year, 1, 1, 12, 0)
    days = 366 if year%4==0 else 365
    events = []

    # 每日掃描
    # print(f"  正在掃描 {days} 天...") # 減少輸出
    for i in range(days):
        curr_date = start_date + timedelta(days=i)
        transit = create_transit_chart(curr_date, user)
        aspects = calculate_transit_aspects(natal, transit)

        for asp in aspects:
            # 去重邏輯：只存該日最精確的
            events.append({
                'date': curr_date.strftime("%Y-%m-%d"),
                't_planet': asp.p1_name,
                'n_planet': asp.p2_name,
                'aspect': asp.aspect,
                'orb': abs(asp.orbit)
            })

    # 整理報告
    lines = [f"【{year}年 西洋行運報告】", f"姓名: {user['name']}", "="*60]
    lines.append(f"{'日期':<12} {'行運星':<8} {'相位':<6} {'本命星':<8} {'容許度'}")
    lines.append("-" * 50)

    # 簡單去重顯示 (相同相位連續多天，只顯示容許度最小那天，這裡簡化為全部列出但排序)
    events.sort(key=lambda x: (x['date'], x['orb']))

    prev_key = ""
    for e in events:
        key = f"{e['t_planet']}_{e['aspect']}_{e['n_planet']}_{e['date'][:7]}" # 同一個月同一相位只顯示一次
        if key != prev_key:
            t_zh = PLANET_ZH.get(e['t_planet'], e['t_planet'])
            n_zh = PLANET_ZH.get(e['n_planet'], e['n_planet'])
            sym = ASPECT_SYMBOL.get(e['aspect'], e['aspect'])
            lines.append(f"{e['date']:<12} {t_zh:<8} {sym:<6} {n_zh:<8} {e['orb']:.2f}°")
            prev_key = key

    content = "\n".join(lines)
    if export:
        safe_name = safe_filename(user['name'])
        p = PATHS["output_dir"] / safe_name
        p.mkdir(parents=True, exist_ok=True)
        # ★★★ 關鍵修改：強制檔名格式，確保合併時能被識別 ★★★
        (p / f"{safe_name}_西洋行運_{year}.txt").write_text(content, encoding="utf-8")
        print(f"✅ 行運報告 {year} 完成")

# 4. 操作介面 (Cell 4)
這個 Cell 提供了圖形化介面，讓您可以選擇對象、勾選要執行的分析項目 (八字、紫微、占星、流年等)，並一鍵執行。它還包含了一個「自訂備註」的輸入框。

In [4]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime
import sqlite3 # Needed for the new update function

# --- Helper functions (ensure these are defined globally or included here) ---
# Assuming add_user_to_db and get_all_users_from_db are accessible from previous cells.
# If not, they would need to be included here or run before this cell.

# Define update_user_in_db function (moved from thought process for completeness)
def update_user_in_db(user_data: dict):
    if not DB_PATH.exists():
        print(f"⚠️ 錯誤：資料庫檔案 {DB_PATH} 不存在。請先執行 Cell 0 初始化資料庫。")
        return

    if 'username' not in user_data or not user_data['username']:
        print("⚠️ 錯誤：更新使用者時 'username' 為必填！")
        return

    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        try:
            update_fields = []
            update_values = []
            for key, value in user_data.items():
                if key not in ['username', 'id', 'created_at']:# Exclude immutable fields from SET clause
                    update_fields.append(f"{key} = ?")
                    update_values.append(value)
            update_values.append(user_data['username']) # Add username for WHERE clause

            if not update_fields:
                print("ℹ️ 沒有可更新的欄位。")
                return

            sql = f"UPDATE users SET {', '.join(update_fields)} WHERE username = ?"
            cursor.execute(sql, tuple(update_values))
            conn.commit()
            if cursor.rowcount > 0:
                print(f"✅ 使用者 '{user_data['name']}' ({user_data['username']}) 已成功更新！")
            else:
                print(f"⚠️ 警告：找不到使用者 '{user_data['username']}'，未執行更新。")
        except Exception as e:
            print(f"❌ 更新使用者時發生錯誤：{e}")

# New function: add_user_to_db (Moved from previous thought to ensure it's here if not globally available)
def add_user_to_db(user_data: dict):
    if not DB_PATH.exists():
        print(f"⚠️ 錯誤：資料庫檔案 {DB_PATH} 不存在。請先執行 Cell 0 初始化資料庫。")
        return

    required_fields = ['username', 'name', 'gender', 'year', 'month', 'day', 'hour', 'minute']
    for field in required_fields:
        if not user_data.get(field):
            raise ValueError(f"❌ 錯誤：欄位 '{field}' 為必填！")

    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        try:
            # Check if username already exists
            cursor.execute("SELECT id FROM users WHERE username = ?", (user_data['username'],))
            if cursor.fetchone():
                raise ValueError(f"❌ 錯誤：使用者名稱 '{user_data['username']}' 已存在！")

            sql = "INSERT INTO users (username, name, gender, year, month, day, hour, minute, city, lng, lat, tz_str, note) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)"
            values = (
                user_data['username'], user_data['name'], user_data['gender'],
                user_data['year'], user_data['month'], user_data['day'],
                user_data['hour'], user_data['minute'], user_data.get('city'),
                user_data.get('lng'), user_data.get('lat'), user_data.get('tz_str'), user_data.get('note')
            )
            cursor.execute(sql, values)
            conn.commit()
            print(f"✅ 使用者 '{user_data['name']}' ({user_data['username']}) 已成功新增！")
        except sqlite3.IntegrityError as e:
            print(f"❌ 新增使用者失敗 (資料重複或格式錯誤): {e}")
        except Exception as e:
            print(f"❌ 新增使用者失敗: {e}")

# New function: delete_user_from_db
def delete_user_from_db(username: str):
    if not DB_PATH.exists():
        print(f"⚠️ 錯誤：資料庫檔案 {DB_PATH} 不存在。請先執行 Cell 0 初始化資料庫。")
        return

    if not username:
        print("⚠️ 錯誤：刪除使用者時 'username' 為必填！")
        return

    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        try:
            cursor.execute("DELETE FROM users WHERE username = ?", (username,))
            conn.commit()
            if cursor.rowcount > 0:
                print(f"✅ 使用者 '{username}' 已成功從資料庫中移除！")
            else:
                print(f"⚠️ 警告：找不到使用者 '{username}'，未執行刪除。")
        except Exception as e:
            print(f"❌ 刪除使用者時發生錯誤：{e}")

# UI 元件
db_selector = widgets.SelectMultiple(options=list(USER_DB.keys()), description='選擇對象:', layout=widgets.Layout(width='98%', height='120px'))
start_y = widgets.IntText(value=datetime.now().year, description='起始年份:')
end_y = widgets.IntText(value=datetime.now().year + 1, description='結束年份:')

chk_bazi = widgets.Checkbox(value=False, description='八字')
chk_ziwei_natal = widgets.Checkbox(value=False, description='紫微本命')
chk_ziwei_year = widgets.Checkbox(value=False, description='紫微流年')
chk_ziwei_month = widgets.Checkbox(value=False, description='紫微流月')
chk_astro_natal = widgets.Checkbox(value=False, description='西洋本命')
chk_astro_transit = widgets.Checkbox(value=False, description='西洋行運')
chk_merge = widgets.Checkbox(value=True, description='自動合併報告')

txt_custom = widgets.Textarea(description='自訂備註:', placeholder='輸入想加入報告結尾的文字...', layout=widgets.Layout(width='98%', height='100px'))
btn_run = widgets.Button(description='🚀 開始生產', button_style='success', layout=widgets.Layout(width='98%'))
out_log = widgets.Output(layout={'border': '1px solid #ccc', 'height': '300px', 'overflow_y': 'scroll'})

all_funcs = [chk_bazi, chk_ziwei_natal, chk_ziwei_year, chk_ziwei_month, chk_astro_natal, chk_astro_transit, chk_merge]

# 新增人物資料輸入元件
new_username_input = widgets.Text(description='帳號:')
new_name_input = widgets.Text(description='姓名:')
new_gender_input = widgets.Dropdown(options=['男', '女', '其他'], value='男', description='性別:')
new_year_input = widgets.IntText(value=datetime.now().year, description='年份:')
new_month_input = widgets.IntText(value=1, description='月份:')
new_day_input = widgets.IntText(value=1, description='日期:')
new_hour_input = widgets.IntText(value=0, description='時辰:')
new_minute_input = widgets.IntText(value=0, description='分鐘:')
new_city_input = widgets.Text(description='城市:')
new_lng_input = widgets.FloatText(description='經度:')
new_lat_input = widgets.FloatText(description='緯度:')
new_tz_str_input = widgets.Text(description='時區字串:')
new_note_input = widgets.Textarea(description='備註:', layout=widgets.Layout(width='98%', height='60px'))
add_user_btn = widgets.Button(description='➕ 新增人物', button_style='info', layout=widgets.Layout(width='32%')) # Adjusted width
update_user_btn = widgets.Button(description='🔄 更新人物', button_style='warning', layout=widgets.Layout(width='32%')) # New button
delete_user_btn = widgets.Button(description='🗑️ 刪除人物', button_style='danger', layout=widgets.Layout(width='32%')) # New delete button

# 城市自動填入功能
def on_city_input_change(change):
    with out_log:
        city_name = change.new
        if city_name and city_name in TAIWAN_CITY_COORDS:
            coords = TAIWAN_CITY_COORDS[city_name]
            new_lng_input.value = coords['lng']
            new_lat_input.value = coords['lat']
            new_tz_str_input.value = coords['tz_str']
            print(f"ℹ️ 城市 '{city_name}' 資訊已自動填入。")
        else:
            new_lng_input.value = 0.0
            new_lat_input.value = 0.0
            new_tz_str_input.value = ''
            if city_name: print(f"⚠️ 城市 '{city_name}' 未在預設列表中找到，請手動輸入經緯度/時區。")

new_city_input.observe(on_city_input_change, names='value')

# DB Selector change handler
def on_db_selector_change(change):
    with out_log:
        selected_users = change.new
        if len(selected_users) == 1:
            username = selected_users[0]
            user = USER_DB.get(username)
            if user:
                print(f"ℹ️ 已選取使用者 '{username}'，填入資料以便編輯。")
                new_username_input.value = user.get('username', '')
                new_name_input.value = user.get('name', '')
                new_gender_input.value = user.get('gender', '男')
                new_year_input.value = user.get('year', datetime.now().year)
                new_month_input.value = user.get('month', 1)
                new_day_input.value = user.get('day', 1)
                new_hour_input.value = user.get('hour', 0)
                new_minute_input.value = user.get('minute', 0)
                new_city_input.value = user.get('city', '')
                new_lng_input.value = user.get('lng', 0.0)
                new_lat_input.value = user.get('lat', 0.0)
                new_tz_str_input.value = user.get('tz_str', '')
                new_note_input.value = user.get('note') or '' # Modified line: Ensure it's always a string
            else:
                print(f"⚠️ 錯誤：選取的使用者 '{username}' 不在資料庫中。")
                # Clear fields if user not found
                new_username_input.value = ''
                new_name_input.value = ''
                new_gender_input.value = '男'
                new_year_input.value = datetime.now().year
                new_month_input.value = 1
                new_day_input.value = 1
                new_hour_input.value = 0
                new_minute_input.value = 0
                new_city_input.value = ''
                new_lng_input.value = 0.0
                new_lat_input.value = 0.0
                new_tz_str_input.value = ''
                new_note_input.value = ''
        else:
            # Clear all input fields if no user or multiple users are selected
            new_username_input.value = ''
            new_name_input.value = ''
            new_gender_input.value = '男'
            new_year_input.value = datetime.now().year
            new_month_input.value = 1
            new_day_input.value = 1
            new_hour_input.value = 0
            new_minute_input.value = 0
            new_city_input.value = ''
            new_lng_input.value = 0.0
            new_lat_input.value = 0.0
            new_tz_str_input.value = ''
            new_note_input.value = ''
            if len(selected_users) > 1: print("⚠️ 請單選一位使用者進行編輯。")
            else: print("ℹ️ 請選取一位使用者以編輯資料，或輸入新資料新增人物。")

db_selector.observe(on_db_selector_change, names='value')

def on_add_user_click(b):
    with out_log:
        clear_output(wait=True)
        print("嘗試新增使用者...")
        user_data = {
            'username': new_username_input.value,
            'name': new_name_input.value,
            'gender': new_gender_input.value,
            'year': new_year_input.value,
            'month': new_month_input.value,
            'day': new_day_input.value,
            'hour': new_hour_input.value,
            'minute': new_minute_input.value,
            'city': new_city_input.value if new_city_input.value else None,
            'lng': new_lng_input.value if new_lng_input.value else None,
            'lat': new_lat_input.value if new_lat_input.value else None,
            'tz_str': new_tz_str_input.value if new_tz_str_input.value else None,
            'note': new_note_input.value if new_note_input.value else None,
        }

        # 簡易驗證
        if not user_data['username'] or not user_data['name']:
            print("❌ 錯誤：帳號和姓名為必填！")
            return

        try:
            add_user_to_db(user_data) # Call the add_user_to_db function
            # Reload USER_DB and update selector options
            global USER_DB
            USER_DB = get_all_users_from_db()
            db_selector.options = list(USER_DB.keys())
            print(f"✅ 使用者 '{user_data['name']}' 已新增並重新載入資料庫。")
            # Clear input fields after successful addition
            new_username_input.value = ''
            new_name_input.value = ''
            new_gender_input.value = '男'
            new_year_input.value = datetime.now().year
            new_month_input.value = 1
            new_day_input.value = 1
            new_hour_input.value = 0
            new_minute_input.value = 0
            new_city_input.value = ''
            new_lng_input.value = 0.0
            new_lat_input.value = 0.0
            new_tz_str_input.value = ''
            new_note_input.value = ''
        except Exception as e:
            print(f"❌ 新增使用者失敗: {e}")

def on_update_user_click(b):
    with out_log:
        clear_output(wait=True)
        print("嘗試更新使用者...")
        user_data = {
            'username': new_username_input.value,
            'name': new_name_input.value,
            'gender': new_gender_input.value,
            'year': new_year_input.value,
            'month': new_month_input.value,
            'day': new_day_input.value,
            'hour': new_hour_input.value,
            'minute': new_minute_input.value,
            'city': new_city_input.value if new_city_input.value else None,
            'lng': new_lng_input.value if new_lng_input.value else None,
            'lat': new_lat_input.value if new_lat_input.value else None,
            'tz_str': new_tz_str_input.value if new_tz_str_input.value else None,
            'note': new_note_input.value if new_note_input.value else None,
        }

        if not user_data['username']:
            print("❌ 錯誤：更新使用者時必須指定 '帳號'！")
            return

        try:
            update_user_in_db(user_data) # Call the new update_user_in_db function
            global USER_DB
            USER_DB = get_all_users_from_db()
            db_selector.options = list(USER_DB.keys())
            # No need to clear fields, user might want to continue editing
        except Exception as e:
            print(f"❌ 更新使用者失敗: {e}")

def on_delete_user_click(b):
    with out_log:
        clear_output(wait=True)
        selected_users = db_selector.value
        if not selected_users:
            print("⚠️ 請先從列表中選擇要刪除的人物！")
            return
        if len(selected_users) > 1:
            print("⚠️ 請單選一位使用者進行刪除。")
            return

        username_to_delete = selected_users[0]
        print(f"嘗試刪除使用者 '{username_to_delete}'...")

        try:
            delete_user_from_db(username_to_delete) # Call the new delete_user_from_db function
            global USER_DB
            USER_DB = get_all_users_from_db()
            db_selector.options = list(USER_DB.keys())
            db_selector.value = () # Clear selection
            # Clear input fields after deletion
            new_username_input.value = ''
            new_name_input.value = ''
            new_gender_input.value = '男'
            new_year_input.value = datetime.now().year
            new_month_input.value = 1
            new_day_input.value = 1
            new_hour_input.value = 0
            new_minute_input.value = 0
            new_city_input.value = ''
            new_lng_input.value = 0.0
            new_lat_input.value = 0.0
            new_tz_str_input.value = ''
            new_note_input.value = ''
        except Exception as e:
            print(f"❌ 刪除使用者失敗: {e}")

def on_run_click(b):
    with out_log:
        clear_output(wait=True)
        users = db_selector.value
        if not users: print("⚠️ 請選擇對象！"); return

        years = list(range(start_y.value, end_y.value + 1))

        print(f"📋 開始處理 {len(users)} 位使用者...")
        for uid in users:
            print(f"\n🔹 處理: {uid}")
            try:
                if chk_bazi.value: run_bazi_analysis(uid)
                if chk_ziwei_natal.value: run_ziwei_analysis(uid)
                if chk_astro_natal.value: run_western_astrology(uid)

                for y in years:
                    if chk_ziwei_year.value: run_ziwei_annual_transit(uid, y)
                    if chk_ziwei_month.value:
                        for m in range(1, 13): run_ziwei_monthly_transit(uid, y, m)
                    if chk_astro_transit.value: run_western_transit(uid, y)

                if chk_merge.value: merge_user_reports(uid, custom_text=txt_custom.value)
            except Exception as e:
                print(f"❌ 錯誤: {e}")
                import traceback
                traceback.print_exc()
        print("\n✅ 全部完成！")

btn_run.on_click(on_run_click)
add_user_btn.on_click(on_add_user_click)
update_user_btn.on_click(on_update_user_click)
delete_user_btn.on_click(on_delete_user_click) # Attach handler for delete button

# 新增人物UI排版
add_user_ui_content = widgets.VBox([
    widgets.HBox([new_username_input, new_name_input]),
    widgets.HBox([new_gender_input, new_year_input, new_month_input, new_day_input]),
    widgets.HBox([new_hour_input, new_minute_input]),
    widgets.HBox([new_city_input, new_lng_input, new_lat_input]),
    new_tz_str_input,
    new_note_input,
    widgets.HBox([add_user_btn, update_user_btn, delete_user_btn]) # Combined buttons with delete
])

add_user_accordion = widgets.Accordion(children=[add_user_ui_content])
add_user_accordion.set_title(0, '➕ 新增/編輯/刪除人物資料') # Updated title

# 主要UI排版
ui = widgets.VBox([
    widgets.HTML("<h3>🔮 命理批次生成系統</h3>"),
    add_user_accordion, # 將新增人物的Accordion加入主UI
    widgets.HTML("<h4>分析選項</h4>"),
    db_selector,
    widgets.HBox([widgets.Label("年份:"), start_y, end_y]),
    widgets.HBox([chk_bazi, chk_ziwei_natal, chk_astro_natal]),
    widgets.HBox([chk_ziwei_year, chk_ziwei_month, chk_astro_transit, chk_merge]),
    txt_custom,
    btn_run,
    out_log
])
display(ui)

# 結束

In [6]:
from kerykeion import to_context

subject = AstrologicalSubjectFactory.from_birth_data(
    "Example", 1990, 6, 15, 14, 30,
    lng=12.4964, lat=41.9028, tz_str="Europe/Rome", online=False
)
chart_data = ChartDataFactory.create_natal_chart_data(subject)

ai_text = to_context(chart_data)
# Non-qualitative, factual description for LLM processing

In [7]:
from kerykeion import AstrologicalSubjectFactory, to_context

# 創建一個占星主題
subject = AstrologicalSubjectFactory.from_birth_data(
    "Omi", 1986, 5, 1, 00, 40, "Taipei", "TW"
)

# 生成 AI 可處理的上下文
context = to_context(subject)

# 打印上下文
print(context)

********
NO GEONAMES USERNAME SET!
Using the default geonames username is not recommended, please set a custom one!
You can get one for free here:
https://www.geonames.org/login
Keep in mind that the default username is limited to 2000 requests per hour and is shared with everyone else using this library.
********


Chart for "Omi"
Birth data: 1986-05-01 00:40, Taipei, TW
Coordinates: 25.05°N, 121.53°E
Timezone: Asia/Taipei
Zodiac system: Tropical
House system: Placidus
Perspective: Apparent Geocentric

Planetary positions:
  - Sun at 10.04° in Taurus in Third House, absolute position 40.04°, quality: Fixed, element: Earth, direct motion, speed 0.9709°/day, declination 14.83°
  - Moon at 4.37° in Aquarius in Twelfth House, absolute position 304.37°, quality: Fixed, element: Air, direct motion, speed 13.7751°/day, declination -24.28°
  - Mercury at 18.32° in Aries in Second House, absolute position 18.32°, quality: Cardinal, element: Fire, direct motion, speed 1.6049°/day, declination 4.77°
  - Venus at 4.75° in Gemini in Fourth House, absolute position 64.75°, quality: Mutable, element: Air, direct motion, speed 1.2168°/day, declination 21.74°
  - Mars at 14.98° in Capricorn in Twelfth House, absolute position 284.98°, quality: Cardinal, element: Earth, direct motion, speed 0.3689°/day, declinatio

In [18]:
# Cell: DestinyCore_v3.2_Path_AutoFix.py
# ============================================================================
# 🔮 Omi 命理核心引擎 v3.2 (路徑暴力搜索修復版)
# ============================================================================

import json
import subprocess
import re
import sys
import os
from pathlib import Path
from datetime import datetime
from google.colab import drive

# --- 1. 強制掛載 Google Drive (確保環境連線) ---
print("🔌 正在檢查 Google Drive 連線...")
if not Path("/content/drive").exists():
    drive.mount('/content/drive')
    print("✅ Google Drive 掛載成功")
else:
    print("✅ Google Drive 已連線")

# --- 2. 暴力搜索 bazi.py (不再猜路徑，直接找) ---
def find_bazi_script(start_path):
    print(f"🕵️‍♂️ 正在 {start_path} 範圍內搜索 'bazi.py'...")
    search_path = Path(start_path)
    if not search_path.exists():
        print(f"❌ 搜索路徑不存在: {search_path}")
        return None

    # 遞迴搜索所有子目錄
    for path in search_path.rglob('bazi.py'):
        if path.is_file():
            print(f"🎯 鎖定目標: {path}")
            return path
    return None

# 設定搜索起點 (針對妳的資料夾結構)
SEARCH_ROOT = "/content/drive/MyDrive/Colab Notebooks/Life_DB"
BAZI_SCRIPT_PATH = find_bazi_script(SEARCH_ROOT)

if not BAZI_SCRIPT_PATH:
    print("😱 崩潰：在 Life_DB 資料夾內找不到 bazi.py！請確認檔案是否已上傳。")
    # 最後一搏：嘗試找更上層
    BAZI_SCRIPT_PATH = find_bazi_script("/content/drive/MyDrive")

# --- 3. 套件檢查 ---
try:
    from kerykeion import AstrologicalSubjectFactory, ChartDataFactory, to_context
    print("✅ Kerykeion (西洋占星) 就緒")
except ImportError:
    print("⚠️ 警告：缺少 kerykeion，正在嘗試安裝...")
    subprocess.run([sys.executable, "-m", "pip", "install", "kerykeion"], check=True)

try:
    from py_iztro import Astro
    HAS_PY_IZTRO = True
    print("✅ Py-Iztro (紫微斗數) 就緒")
except ImportError:
    print("⚠️ 警告：缺少 py-iztro，正在嘗試安裝...")
    subprocess.run([sys.executable, "-m", "pip", "install", "py-iztro"], check=True)
    from py_iztro import Astro # 安裝後重新匯入
    HAS_PY_IZTRO = True

# --- 4. 內建修正與資料庫 ---
CUSTOM_FIXES = {
    "天幹": "天干", "天乾": "天干", "醜": "丑", "枭": "梟", "财": "財", "伤": "傷",
    "阳": "陽", "阴": "陰", "宫": "宮", "运": "運", "岁": "歲", "华盖": "華蓋",
    "贪狼": "貪狼", "廉贞": "廉貞", "鬥數": "斗數",
}

TAIWAN_CITY_COORDS = {
    "台北": {"lng": 121.5654, "lat": 25.0330, "tz_str": "Asia/Taipei"},
    "新北": {"lng": 121.4628, "lat": 25.0146, "tz_str": "Asia/Taipei"},
    "桃園": {"lng": 121.3009, "lat": 24.9936, "tz_str": "Asia/Taipei"},
    "台中": {"lng": 120.6736, "lat": 24.1477, "tz_str": "Asia/Taipei"},
    "高雄": {"lng": 120.3014, "lat": 22.6273, "tz_str": "Asia/Taipei"},
}

# --- 5. 核心引擎 (修正版) ---
class DestinyCore:
    def __init__(self, username, user_db):
        if username not in user_db:
            raise ValueError(f"❌ 找不到使用者: {username}")

        self.raw_user = user_db[username]
        self.name = self.raw_user['name']
        self.dt = datetime(
            self.raw_user['year'], self.raw_user['month'], self.raw_user['day'],
            self.raw_user['hour'], self.raw_user['minute']
        )

        # 座標補全
        self.city = self.raw_user.get('city', '台北')
        if 'lng' not in self.raw_user:
            coord = TAIWAN_CITY_COORDS.get(self.city, TAIWAN_CITY_COORDS["台北"])
            self.lng = coord['lng']
            self.lat = coord['lat']
            self.tz_str = coord.get('tz_str', "Asia/Taipei")
        else:
            self.lng = self.raw_user['lng']
            self.lat = self.raw_user['lat']
            self.tz_str = self.raw_user.get('tz_str', "Asia/Taipei")

    def _fix_text(self, text):
        if not text: return ""
        for wrong, correct in CUSTOM_FIXES.items():
            text = text.replace(wrong, correct)
        return text

    def get_western_data(self):
        try:
            subject = AstrologicalSubjectFactory.from_birth_data(
                name=self.name,
                year=self.dt.year, month=self.dt.month, day=self.dt.day,
                hour=self.dt.hour, minute=self.dt.minute,
                lng=self.lng, lat=self.lat, tz_str=self.tz_str,
                online=False
            )
            chart = ChartDataFactory.create_natal_chart_data(subject)
            return to_context(chart)
        except Exception as e:
            return f"Western Error: {e}"

    def get_bazi_data(self):
        if not BAZI_SCRIPT_PATH:
            return "❌ Bazi Error: 找不到 bazi.py 檔案，無法執行。"

        # 這裡改用 sys.executable 確保用的是 Colab 當前的 python 環境
        cmd = [sys.executable, str(BAZI_SCRIPT_PATH),
               str(self.dt.year), str(self.dt.month), str(self.dt.day), str(self.dt.hour), '-g']
        if self.raw_user['gender'] == "女":
            cmd.append("-n")

        try:
            result = subprocess.run(
                cmd,
                capture_output=True, text=True, check=True, encoding='utf-8',
                cwd=BAZI_SCRIPT_PATH.parent # 重要：在腳本所在目錄執行
            )
            raw_text = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])').sub('', result.stdout)
            return self._fix_text(raw_text.strip())
        except subprocess.CalledProcessError as e:
            return f"❌ Bazi Execution Failed: {e.stderr}"
        except Exception as e:
            return f"❌ Bazi Unknown Error: {e}"

    def get_ziwei_data(self):
        if not HAS_PY_IZTRO: return "Ziwei Error: Module missing"
        try:
            gender = "男" if self.raw_user['gender'] in ["M", "male", "男"] else "女"
            time_idx = 0 if self.dt.hour == 23 else (self.dt.hour + 1) // 2

            astro = Astro().by_solar(
                f"{self.dt.year}-{self.dt.month}-{self.dt.day}",
                time_idx, gender, language="zh-TW"
            )
            data = json.loads(astro.model_dump_json(by_alias=True))

            output = {
                "格局": {
                    "五行局": data.get("fiveElementsClass"),
                    "命主": data.get("soul"),
                    "身主": data.get("body")
                },
                "十二宮位": {}
            }

            for p in data.get("palaces", []):
                palace_name = self._fix_text(p["name"])
                # 這裡保留了防止 NoneType 錯誤的邏輯
                output["十二宮位"][palace_name] = {
                    "干支": p['heavenlyStem'] + p['earthlyBranch'],
                    "主星": [self._fix_text(s['name']) + f"({s.get('brightness','')})" + (f"[生年{s['mutagen']}]" if s.get('mutagen') else "") for s in p.get("majorStars", [])],
                    "四化": [self._fix_text(s['name']) + "化" + s['mutagen'] for s in p.get("majorStars", []) if s.get('mutagen')] + \
                            [self._fix_text(s['name']) + "化" + s['mutagen'] for s in p.get("minorStars", []) if s.get('mutagen')],
                    "輔星": [self._fix_text(s['name']) for s in p.get("minorStars", []) if s.get("importance") != "low"],
                    "神煞": [self._fix_text(s['name']) for s in p.get("years", [])]
                }
            return json.dumps(output, ensure_ascii=False, indent=2)
        except Exception as e:
            return f"Ziwei Logic Error: {e}"

    def generate_report(self):
        return {
            "Meta": {"Name": self.name, "Birth": str(self.dt), "Note": self.raw_user.get('note','')},
            "Western": self.get_western_data(),
            "Bazi": self.get_bazi_data(),
            "Ziwei": self.get_ziwei_data()
        }

# --- 6. 執行區 ---
USER_DB = {
    "omi": {
        "name": "omi", "gender": "女", "year": 1986, "month": 5, "day": 1, "hour": 0, "minute": 40,
        "city": "台北", "lng": 121.5654, "lat": 25.0330, "note": "本人",
    },
    "蔡佳惠": {
        "name": "蔡佳惠", "gender": "女", "year": 1988, "month": 4, "day": 18, "hour": 21, "minute": 50,
        "city": "新北", "lng": 121.4628, "lat": 25.0146, "note": "林子歡同學",
    }
}

final_output = {}
print("\n⚡ 開始執行 DestinyCore v3.2 (Path-AutoFix)...")

if BAZI_SCRIPT_PATH:
    print(f"📝 八字腳本狀態: 鎖定 ({BAZI_SCRIPT_PATH})")
else:
    print("🚨 八字腳本狀態: 未找到 (八字功能將回傳錯誤)")

for uid in USER_DB:
    try:
        core = DestinyCore(uid, USER_DB)
        final_output[uid] = core.generate_report()
        print(f"✅ {uid} 生成成功")
    except Exception as e:
        print(f"❌ {uid} 失敗: {e}")

print("\n" + "="*30)
print("請檢查下方 JSON，八字區塊不應再出現 [Error]：")
print("="*30)
print(json.dumps(final_output, indent=2, ensure_ascii=False))

🔌 正在檢查 Google Drive 連線...
✅ Google Drive 已連線
🕵️‍♂️ 正在 /content/drive/MyDrive/Colab Notebooks/Life_DB 範圍內搜索 'bazi.py'...
🎯 鎖定目標: /content/drive/MyDrive/Colab Notebooks/Life_DB/bazi-master/bazi.py
✅ Kerykeion (西洋占星) 就緒
✅ Py-Iztro (紫微斗數) 就緒

⚡ 開始執行 DestinyCore v3.2 (Path-AutoFix)...
📝 八字腳本狀態: 鎖定 (/content/drive/MyDrive/Colab Notebooks/Life_DB/bazi-master/bazi.py)
✅ omi 生成成功
✅ 蔡佳惠 生成成功

請檢查下方 JSON，八字區塊不應再出現 [Error]：
{
  "omi": {
    "Meta": {
      "Name": "omi",
      "Birth": "1986-05-01 00:40:00",
      "Note": "本人"
    },
    "Western": "Natal Chart Analysis\n==================================================\n\nChart for \"omi\"\nBirth data: 1986-05-01 00:40, Greenwich, GB\nCoordinates: 25.03°N, 121.57°E\nTimezone: Asia/Taipei\nZodiac system: Tropical\nHouse system: Placidus\nPerspective: Apparent Geocentric\n\nPlanetary positions:\n  - Sun at 10.04° in Taurus in Third House, absolute position 40.04°, quality: Fixed, element: Earth, direct motion, speed 0.9709°/day, declination 14.83°

In [19]:
# Cell: DestinyCore_Dashboard_GUI.py
# ============================================================================
# 🔮 DestinyCore 戰略指揮中心 (前端視覺化介面)
# ============================================================================

import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import json

# --- 1. 樣式設定 (美學至上) ---
STYLE_HEADER = "background-color: #2c3e50; color: white; padding: 10px; border-radius: 5px; font-size: 16px; font-weight: bold;"
STYLE_BTN = "font-weight: bold;"

# --- 2. 輔助顯示函式 ---
def format_ziwei_grid(json_str):
    """ 將紫微 JSON 字串轉換為易讀的宮位卡片 """
    try:
        if not json_str or "Error" in json_str:
            return f"❌ 紫微數據錯誤: {json_str}"

        data = json.loads(json_str)

        # 顯示格局
        pattern = data.get("格局", {})
        html = f"<h4>🏗️ 格局：{pattern.get('五行局')} | 命主：{pattern.get('命主')} | 身主：{pattern.get('身主')}</h4>"
        html += "<div style='display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px;'>"

        # 顯示 12 宮位
        palaces = data.get("十二宮位", {})
        # 定義顯示順序 (命宮起逆行或順行皆可，這裡簡單列出)
        order = ['命宮', '兄弟', '夫妻', '子女', '財帛', '疾厄', '遷移', '僕役', '官祿', '田宅', '福德', '父母']

        for name in order:
            if name not in palaces: continue
            p = palaces[name]

            # 宮位卡片樣式
            card_style = "border: 1px solid #ddd; padding: 8px; border-radius: 4px; background: #f9f9f9;"
            if "命宮" in name: card_style += "border-left: 5px solid #e74c3c;" # 命宮標紅
            elif "身宮" in p.get("宮位性質", ""): card_style += "border-left: 5px solid #f39c12;" # 身宮標黃

            # 星曜格式化
            stars_html = ""
            if p['主星']: stars_html += f"<div style='color:#c0392b; font-weight:bold;'>{' '.join(p['主星'])}</div>"
            if p['四化']: stars_html += f"<div style='color:#8e44ad;'>{' '.join(p['四化'])}</div>"
            if p['輔星']: stars_html += f"<div style='color:#27ae60; font-size:0.9em;'>{' '.join(p['輔星'])}</div>"

            html += f"""
            <div style="{card_style}">
                <div style="border-bottom:1px solid #eee; margin-bottom:5px;">
                    <b>{name}</b> <span style="color:#7f8c8d; font-size:0.8em;">({p['干支']})</span>
                </div>
                {stars_html}
            </div>
            """
        html += "</div>"
        return widgets.HTML(html)
    except Exception as e:
        return widgets.HTML(f"⚠️ 解析失敗: {e}")

# --- 3. 主介面類別 ---
class DestinyDashboard:
    def __init__(self, user_db):
        self.user_db = user_db
        self.core = None

        # UI 元件
        self.title = widgets.HTML(f"<div style='{STYLE_HEADER}'>🔮 DestinyCore 戰略指揮中心 v3.2</div>")

        # 選擇器
        self.dropdown = widgets.Dropdown(
            options=[(f"{v['name']} ({k})", k) for k, v in user_db.items()],
            description='目標對象:',
            style={'description_width': 'initial'}
        )

        # 按鈕
        self.btn_run = widgets.Button(
            description='🚀 啟動全系統掃描',
            button_style='primary', # 'success', 'info', 'warning', 'danger' or ''
            layout=widgets.Layout(width='100%', margin='10px 0')
        )
        self.btn_run.on_click(self.on_run)

        # 輸出區域 (Tab 分頁)
        self.out_profile = widgets.Output()
        self.out_western = widgets.Output()
        self.out_bazi = widgets.Output()
        self.out_ziwei = widgets.Output()
        self.out_log = widgets.Output(layout={'border': '1px solid #eee', 'height': '100px', 'overflow_y': 'scroll'})

        self.tabs = widgets.Tab(children=[self.out_profile, self.out_western, self.out_bazi, self.out_ziwei])
        self.tabs.set_title(0, '📋 基本檔案')
        self.tabs.set_title(1, '🌌 西洋占星')
        self.tabs.set_title(2, '🏮 八字命理')
        self.tabs.set_title(3, '🔯 紫微斗數')

        # 組合介面
        self.ui = widgets.VBox([
            self.title,
            widgets.HBox([self.dropdown]),
            self.btn_run,
            self.out_log,
            self.tabs
        ])

    def log(self, msg):
        with self.out_log:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

    def on_run(self, b):
        self.out_profile.clear_output()
        self.out_western.clear_output()
        self.out_bazi.clear_output()
        self.out_ziwei.clear_output()
        self.out_log.clear_output()

        target_uid = self.dropdown.value
        self.log(f"正在載入 {target_uid} 的數據模組...")

        try:
            # 1. 呼叫後端 (DestinyCore)
            self.core = DestinyCore(target_uid, self.user_db)
            report_raw = self.core.generate_report()

            # 由於 generate_report 返回的是 JSON 字串，我們轉回 Dict 方便處理
            # (如果妳的 v3.2 返回的是 dict 則不用 loads，這邊假設是 dict)
            if isinstance(report_raw, str):
                data = json.loads(report_raw)
            else:
                data = report_raw

            self.log("數據生成完畢，正在渲染 UI...")

            # 2. 渲染基本檔案
            with self.out_profile:
                meta = data.get("Meta", {}) or data.get("Info", {})
                display(widgets.HTML(f"""
                <h3>{meta.get('Name')}</h3>
                <ul>
                    <li><b>生日：</b> {meta.get('Birth')}</li>
                    <li><b>備註：</b> {meta.get('Note')}</li>
                </ul>
                """))

            # 3. 渲染西洋占星 (Markdown 文本)
            with self.out_western:
                western_txt = data.get("Western", "") or data.get("Western_Context", "")
                print(western_txt) # 直接 print 保持格式

            # 4. 渲染八字 (Preformatted Text)
            with self.out_bazi:
                bazi_txt = data.get("Bazi", "") or data.get("Bazi_Data", "")
                # 八字排盤需要等寬字體才好看
                display(widgets.HTML(f"<pre style='font-family: monospace; white-space: pre-wrap;'>{bazi_txt}</pre>"))

            # 5. 渲染紫微 (Grid View)
            with self.out_ziwei:
                ziwei_json = data.get("Ziwei", "") or data.get("Ziwei_Data", "")
                display(format_ziwei_grid(ziwei_json))

            self.log("✅ 任務完成！")

        except Exception as e:
            self.log(f"❌ 發生錯誤: {e}")
            import traceback
            with self.out_log:
                traceback.print_exc()

# --- 4. 啟動 ---
# 確保 USER_DB 存在 (從上一個 Cell 繼承)
if 'USER_DB' in globals():
    dashboard = DestinyDashboard(USER_DB)
    display(dashboard.ui)
else:
    print("❌ 找不到 USER_DB，請先執行上面的 DestinyCore 代碼塊！")